# 05 · how machines read and talk

**before you start:** *Runtime → Change runtime type → T4 GPU → Save.* it's free, and it makes the AI models in this notebook about 20 times faster.

**how to use this:** press ▶ on each cell, top to bottom. anything in a box on the right is yours to change. change it, press ▶ again, see what happens. you can't break anything. if it gets weird: *Runtime → Restart session* and start from the top.

text is harder than photos: there are no pixels. so first the machine chops text into **tokens** and turns them into numbers. after that it's the same idea as everything else this week.

In [ ]:
#@title ▶ run me first (loads the tools)
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM, AutoModelForSeq2SeqLM
import torch, pandas as pd
import warnings; warnings.filterwarnings("ignore")
device = "cuda" if torch.cuda.is_available() else "cpu"
CHAT_MODEL = "Qwen/Qwen2.5-1.5B-Instruct" if device == "cuda" else "Qwen/Qwen2.5-0.5B-Instruct"   # the bigger one plays characters better, but needs the GPU
print(f"ready ✅  running on {'a GPU 🚀' if device == 'cuda' else 'the CPU: works, but switch to a T4 GPU for the chatbot (see the note above)'}")

## 1 · how does it feel? (sentiment)

In [ ]:
#@title one sentence
sentence = "the chai at the canteen today was honestly amazing" #@param {type:"string"}
feel = pipeline("sentiment-analysis", model="distilbert/distilbert-base-uncased-finetuned-sst-2-english")
r = feel(sentence)[0]
print(f"{'😊' if r['label'] == 'POSITIVE' else '😠'}  {r['label'].lower()}  ({r['score']:.0%} sure)")

In [ ]:
#@title a whole pile of reviews
reviews = "best biryani in town | waited 45 minutes for cold food | not bad, not great | the waiter was lovely but the food was terrible | would come back tomorrow" #@param {type:"string"}
texts = [t.strip() for t in reviews.split("|")]
pd.DataFrame([{"review": t, "feeling": r["label"].lower(), "sure": f"{r['score']:.0%}"} for t, r in zip(texts, feel(texts))])

separate reviews with `|`. try sarcasm: *"oh great, another monday"*. it struggles, like people do over text.

## 2 · sort text into your own boxes

In [ ]:
#@title zero-shot classification
text = "my order arrived two days late and the box was crushed" #@param {type:"string"}
boxes = "delivery problem, damaged item, wrong item, billing problem, compliment" #@param {type:"string"}
sorter = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")
r = sorter(text, candidate_labels=[b.strip() for b in boxes.split(",")], multi_label=True)
for label, score in zip(r["labels"], r["scores"]):
    print(f"{score:6.1%}  {label}")

nobody trained this on *your* boxes. it read enough text to guess what the words mean. companies use exactly this to sort thousands of complaints a day.

## 3 · translation

In [ ]:
#@title translate from english
english = "machine learning is just learning from examples" #@param {type:"string"}
to = "hindi" #@param ["hindi", "malayalam", "french", "german", "spanish"]
code = {"hindi": "hi", "malayalam": "ml", "french": "fr", "german": "de", "spanish": "es"}[to]
mt_tok = AutoTokenizer.from_pretrained(f"Helsinki-NLP/opus-mt-en-{code}")
mt = AutoModelForSeq2SeqLM.from_pretrained(f"Helsinki-NLP/opus-mt-en-{code}")
out = mt.generate(**mt_tok([english], return_tensors="pt"), max_new_tokens=200)
print(mt_tok.decode(out[0], skip_special_tokens=True))

if you speak the language, **check it**. small translation models get the gist and miss the grammar. that's what human verification means.

## 4 · tokens: how a model reads

In [ ]:
#@title chop a sentence into tokens
sentence = "unbelievably, the chatbot understood my malayalam-english" #@param {type:"string"}
tok = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")
ids = tok(sentence)["input_ids"]
print(f"{len(sentence.split())} words → {len(ids)} tokens\n")
print(" | ".join(tok.decode([i]) for i in ids))
print("\nand as numbers, which is all the model ever sees:")
print(ids)

## 5 · autocomplete that read the internet
day 1: *"chatgpt is your phone's autocomplete, it just read a lot first."* here's the proof. give it the start of a sentence and see which next words it thinks are likely.

In [ ]:
#@title what comes next?
start = "the best thing about kerala is the" #@param {type:"string"}
lm_tok = AutoTokenizer.from_pretrained(CHAT_MODEL)
lm = AutoModelForCausalLM.from_pretrained(CHAT_MODEL, torch_dtype=torch.float16 if device == "cuda" else torch.float32).to(device)
with torch.no_grad():
    logits = lm(**lm_tok(start, return_tensors="pt").to(device)).logits[0, -1]
probs = torch.softmax(logits.float(), dim=-1)
top = torch.topk(probs, 8)
for p, i in zip(top.values, top.indices):
    print(f"{p.item():6.1%}  {start} **{lm_tok.decode([int(i)]).strip()}**")

that's it. a chatbot picks one of these, adds it, and asks the same question again. a few hundred times. that's a reply.

## 6 · your own chatbot
a small model (1.5 billion weights on the GPU, 0.5 billion on the CPU: chatgpt-sized models have hundreds of billions). you decide **who it is**.

In [ ]:
#@title talk to it
who_it_is = "You are a grumpy Kerala uncle who answers every question with a complaint about the weather, in 3 sentences or less." #@param {type:"string"}
your_message = "what should i study after this course?" #@param {type:"string"}
creativity = 0.7 #@param {type:"slider", min:0.1, max:1.5, step:0.1}
chat = [{"role": "system", "content": who_it_is}, {"role": "user", "content": your_message}]
inputs = lm_tok.apply_chat_template(chat, add_generation_prompt=True, return_tensors="pt", return_dict=True).to(device)
with torch.no_grad():
    out = lm.generate(**inputs, max_new_tokens=150, do_sample=True, temperature=creativity, top_p=0.9)
print(lm_tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))

run it twice: different answer each time (that's the `creativity` dial picking less-likely words). turn creativity up to 1.5 and watch it fall apart.

the **system prompt** (`who_it_is`) is exactly what companies write to turn a general model into *their* customer-care bot.

## 7 · catch it lying

In [ ]:
#@title ask about something that doesn't exist
your_message = "summarise the 2031 IPL final in two sentences" #@param {type:"string"}
chat = [{"role": "system", "content": "You are a helpful assistant."}, {"role": "user", "content": your_message}]
inputs = lm_tok.apply_chat_template(chat, add_generation_prompt=True, return_tensors="pt", return_dict=True).to(device)
with torch.no_grad():
    out = lm.generate(**inputs, max_new_tokens=120, do_sample=False)
print(lm_tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))

the 2031 IPL hasn't happened. watch whether it says so, or **makes something up with a straight face**. that's a hallucination, live.

then try the same question on chatgpt or gemini. bigger models do it less. they still do it.